<!--<badge>--><a href="https://colab.research.google.com/github/JoeChen322/Fintech/blob/main/recommend-system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a><!--</badge>-->


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import TruncatedSVD
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import gdown
import tqdm

%config InlineBackend.figure_format = 'retina'

In [ ]:
url = "https://drive.google.com/uc?id=1TiR645keulkG4pONLpVn-bO4elNBtNpX"
file_path = "Dataset2_Needs.xls"
gdown.download(url, file_path, quiet=False)
needs_df = pd.read_excel(file_path, sheet_name="Needs")
products_df = pd.read_excel(file_path, sheet_name="Products")
metadata_df = pd.read_excel(file_path, sheet_name="Metadata")

print("Needs shape:", needs_df.shape)
print("Products shape:", products_df.shape)
print("Metadata shape:", metadata_df.shape)

Downloading...
From: https://drive.google.com/uc?id=1TiR645keulkG4pONLpVn-bO4elNBtNpX
To: /content/Dataset2_Needs.xls
100%|██████████| 788k/788k [00:00<00:00, 86.6MB/s]


Needs shape: (5000, 10)
Products shape: (11, 3)
Metadata shape: (29, 2)


In [ ]:
needs_df.head()

,ID,Age,Gender,FamilyMembers,FinancialEducation,RiskPropensity,Income,Wealth,IncomeInvestment,AccumulationInvestment
0,1,60,0,2,0.228685,0.233355,68.181525,53.260067,0,1
1,2,78,0,2,0.358916,0.170911,21.807595,135.550048,1,0
2,3,33,1,2,0.317515,0.249703,23.252747,66.303678,0,1
3,4,69,1,4,0.767685,0.654597,166.189034,404.997689,1,1
4,5,58,0,3,0.429719,0.349039,21.186723,58.911930,0,0


In [ ]:
products_df.head()

,IDProduct,Type,Risk
0,1,1,0.55
1,2,0,0.30
2,3,0,0.12
3,4,0,0.44
4,5,1,0.41


In [ ]:
metadata_df.head()

,Metadata,Unnamed: 1
0,Clients,NaN
1,ID,Numerical ID
2,Age,"Age, in years"
3,Gender,"Gender (Female = 1, Male = 0)"
4,FamilyMembers,Number of components


## Data processing

This cell cleans the tabular data, builds the engineered features, and prepares the train/test split used by the recommendation models.


In [ ]:
needs_df = needs_df.copy()
products_df = products_df.copy()
metadata_df = metadata_df.copy()

In [ ]:
needs_df

,ID,Age,Gender,FamilyMembers,FinancialEducation,RiskPropensity,Income,Wealth,IncomeInvestment,AccumulationInvestment
0,1,60,0,2,0.228685,0.233355,68.181525,53.260067,0,1
1,2,78,0,2,0.358916,0.170911,21.807595,135.550048,1,0
2,3,33,1,2,0.317515,0.249703,23.252747,66.303678,0,1
3,4,69,1,4,0.767685,0.654597,166.189034,404.997689,1,1
4,5,58,0,3,0.429719,0.349039,21.186723,58.911930,0,0
...,...,...,...,...,...,...,...,...,...,...
4995,4996,60,1,3,0.609411,0.588353,10.253281,57.368572,0,0
4996,4997,65,1,3,0.523238,0.343272,104.155427,156.824602,1,1
4997,4998,56,1,3,0.433826,0.402771,56.097301,63.283774,0,1
4998,4999,51,1,3,0.559793,0.431419,62.523298,95.357528,0,0


In [ ]:
needs_df.columns

Index(['ID', 'Age', 'Gender', 'FamilyMembers', 'FinancialEducation',
       'RiskPropensity', 'Income ', 'Wealth', 'IncomeInvestment',
       'AccumulationInvestment'],
      dtype='object')

In [ ]:
if "ID" in needs_df.columns:
    needs_df = needs_df.drop(columns=["ID"])

needs_df["Income_log"] = np.log1p(needs_df["Income "].clip(lower=0))
needs_df["Wealth_log"] = np.log1p(needs_df["Wealth"].clip(lower=0))
needs_df["Income_Wealth_Ratio"] = needs_df["Income "] / needs_df["Wealth"].replace(
    0, np.nan
)
needs_df["Income_Wealth_Ratio"] = needs_df["Income_Wealth_Ratio"].fillna(0)

feature_columns = [
    "Age",
    "Gender",
    "FamilyMembers",
    "FinancialEducation",
    "RiskPropensity",
    "Income_log",
    "Wealth_log",
    "Income_Wealth_Ratio",
]
X_base = needs_df[feature_columns].copy()
scaler = MinMaxScaler()
X_base = pd.DataFrame(
    scaler.fit_transform(X_base), columns=feature_columns, index=needs_df.index
)
X_engineered = X_base.copy()

X_train, X_test, y_train_accum, y_test_accum = train_test_split(
    X_base,
    needs_df["AccumulationInvestment"],
    test_size=0.2,
    random_state=42,
    stratify=needs_df["AccumulationInvestment"],
)
_, _, y_train_income, y_test_income = train_test_split(
    X_base,
    needs_df["IncomeInvestment"],
    test_size=0.2,
    random_state=42,
    stratify=needs_df["IncomeInvestment"],
)
test_clients = needs_df.loc[X_test.index].copy()

print("Processed features:", X_base.shape)
print("Train set:", X_train.shape)
print("Test set:", X_test.shape)
X_base.head()


Processed features: (5000, 8)
Train set: (4000, 8)
Test set: (1000, 8)


,Age,Gender,FamilyMembers,FinancialEducation,RiskPropensity,Income_log,Wealth_log,Income_Wealth_Ratio
0,0.531646,0.0,0.25,0.222172,0.243105,0.664782,0.468132,0.006922
1,0.759494,0.0,0.25,0.372410,0.170321,0.441614,0.600160,0.000789
2,0.189873,1.0,0.25,0.324649,0.262161,0.453970,0.498951,0.001829
3,0.645570,1.0,0.75,0.843975,0.734110,0.842246,0.756044,0.002156
4,0.506329,0.0,0.50,0.454090,0.377948,0.436064,0.482307,0.001878


In [ ]:
accumulation_products = products_df[products_df["Type"] == 1].copy()
income_products = products_df[products_df["Type"] == 0].copy()
test_clients = needs_df.loc[X_test.index].copy()

xgb_accum = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="logloss",
    random_state=42,
)
xgb_income = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="logloss",
    random_state=42,
)

xgb_accum.fit(X_train, y_train_accum)
xgb_income.fit(X_train, y_train_income)

accum_pred = pd.Series(xgb_accum.predict(X_test).astype(int), index=X_test.index)
income_pred = pd.Series(xgb_income.predict(X_test).astype(int), index=X_test.index)

print("Accumulation positive predictions:", int(accum_pred.sum()))
print("Income positive predictions:", int(income_pred.sum()))

Accumulation positive predictions: 444
Income positive predictions: 69


In [ ]:
age_min = needs_df["Age"].min()
age_max = needs_df["Age"].max()
wealth_min = needs_df["Wealth"].min()
wealth_max = needs_df["Wealth"].max()

# recommendation Systems

This section adds SVD and autoencoder recommenders built from a client-product interaction matrix.


## SVD


SVD (Singular Value Decomposition) recommends items by factorizing the user-item interaction matrix $R$ into low-dimensional latent factors that capture hidden preferences and item characteristics.

In practice, we approximate:

$$
R \approx U_k \Sigma_k V_k^T
$$

where $k$ is the number of retained latent dimensions.

The predicted preference score for user $u$ and item $i$ is reconstructed from these latent factors (dot product in latent space). Higher reconstructed scores indicate stronger recommendation relevance.

Why this works: SVD compresses noisy sparse interactions into a compact structure, keeping the strongest collaborative signals while removing less informative variance.

This algorithm includes the following steps:

1. Construct user-product interaction matrix $R$
2. Using Truncated SVD for dimensionality reduction
3. Refactor the scoring matrix and generate recommendations
4. Integrated business rules


#### The user-item matrix


We use model-based collaborative filtering algorithm with predicted probability matrix.

This method leverages the **output probabilities** of the already-trained supervised classifiers to construct a dense user–item interaction matrix. Instead of relying on manually crafted scoring rules, we treat the model's predicted propensity for each investment need as a **data-driven confidence score**. This matrix then serves as input to a collaborative filtering algorithm (Truncated SVD) for product recommendation.

Let:

- $P_{\text{acc}}(i) = \mathbb{P}(\text{AccumulationInvestment}=1 \mid \text{client } i)$
- $P_{\text{inc}}(i) = \mathbb{P}(\text{IncomeInvestment}=1 \mid \text{client } i)$

For each client $i$ and product $j$ with product type $\text{Type}(j) \in \{0,1\}$ (0 = Income, 1 = Accumulation), the matrix entry $R_{ij}$ is defined as:

$$
R_{ij} =
\begin{cases}
P_{\text{acc}}(i) \cdot \big(1 - \alpha \cdot |\,\text{RiskPropensity}(i) - \text{Risk}(j)\,|\big), & \text{if Type}(j) = 1 \\[6pt]
P_{\text{inc}}(i) \cdot \big(1 - \alpha \cdot |\,\text{RiskPropensity}(i) - \text{Risk}(j)\,|\big), & \text{if Type}(j) = 0
\end{cases}
$$

where $\alpha \in [0,1]$ is an optional decay factor that penalises risk misalignment. Setting $\alpha = 0$ reduces the matrix to pure predicted probabilities.


In [ ]:
# define decay factor for risk penalty
alpha = 0.35

all_clients = needs_df.copy()
all_features = X_base.copy()

accum_proba = pd.Series(
    xgb_accum.predict_proba(all_features)[:, 1], index=all_features.index, name="P_acc"
)
income_proba = pd.Series(
    xgb_income.predict_proba(all_features)[:, 1], index=all_features.index, name="P_inc"
)

product_meta = products_df.copy()
product_ids = product_meta["IDProduct"].astype(int).tolist()

interaction_matrix = pd.DataFrame(
    index=all_features.index, columns=product_ids, dtype=float
)


# calculate risk penalty and final scores for each product
for _, product in product_meta.iterrows():
    pid = int(product["IDProduct"])
    product_type = int(product["Type"])
    risk = float(product["Risk"])
    risk_penalty = (1.0 - alpha * np.abs(all_clients["RiskPropensity"] - risk)).clip(
        lower=0.0
    )
    base_score = accum_proba if product_type == 1 else income_proba
    interaction_matrix[pid] = (base_score * risk_penalty).astype(float)

n_components = max(
    2, min(6, interaction_matrix.shape[1] - 1, interaction_matrix.shape[0] - 1)
)

### Truncated SVD


In [ ]:
svd = TruncatedSVD(n_components=n_components, random_state=42)
latent_users = svd.fit_transform(interaction_matrix.values)
latent_items = svd.components_

reconstructed_scores = np.dot(latent_users, latent_items)
svd_reconstructed = pd.DataFrame(
    reconstructed_scores,
    index=interaction_matrix.index,
    columns=interaction_matrix.columns,
)

# Business-rule adjustment (age/wealth suitability)
wealth_median = all_clients["Wealth"].median()
for _, product in product_meta.iterrows():
    pid = int(product["IDProduct"])
    product_type = int(product["Type"])
    product_risk = float(product["Risk"])

    rule_multiplier = pd.Series(1.0, index=svd_reconstructed.index)

    if product_type == 1:
        rule_multiplier *= np.where(all_clients["Age"] > 70, 0.85, 1.0)
    else:
        rule_multiplier *= np.where(all_clients["Age"] < 30, 0.90, 1.0)

    rule_multiplier *= np.where(
        (all_clients["Wealth"] < wealth_median) & (product_risk > 0.60),
        0.80,
        1.0,
    )

    rule_multiplier *= np.where(
        np.abs(all_clients["RiskPropensity"] - product_risk) > 0.45,
        0.75,
        1.0,
    )

    svd_reconstructed[pid] = (svd_reconstructed[pid] * rule_multiplier).clip(lower=0.0)

# Build top-3 recommendations per client
client_ids = (svd_reconstructed.index + 1).astype(int)
recommendations = []
for row_idx, client_id in zip(svd_reconstructed.index, client_ids):
    top_products = svd_reconstructed.loc[row_idx].sort_values(ascending=False).head(3)
    for product_id, score in top_products.items():
        recommendations.append(
            {
                "ClientID": int(client_id),
                "ProductID": int(product_id),
                "Score": float(score),
            }
        )

svd_recommendations = pd.DataFrame(recommendations).sort_values(
    ["ClientID", "Score"], ascending=[True, False]
)

print(f"Interaction matrix shape: {interaction_matrix.shape}")
print(f"Retained latent dimensions: {n_components}")

Interaction matrix shape: (5000, 11)
Retained latent dimensions: 6


In [ ]:
svd_recommendations.head()

,ClientID,ProductID,Score
0,1,9,0.960718
1,1,6,0.931829
2,1,5,0.912438
3,2,10,0.464369
4,2,3,0.463652


### Generate recommendations per client


In [ ]:
def recommend_for_client(client_id: int, top_n: int = 3) -> pd.DataFrame:
    client_rows = svd_recommendations[
        svd_recommendations["ClientID"] == int(client_id)
    ].copy()
    if client_rows.empty:
        return pd.DataFrame(columns=["ClientID", "ProductID", "Score", "Type", "Risk"])

    enriched = client_rows.merge(
        products_df[["IDProduct", "Type", "Risk"]],
        left_on="ProductID",
        right_on="IDProduct",
        how="left",
    ).drop(columns=["IDProduct"])

    return enriched.head(top_n)


sample_clients = (X_test.index[:5] + 1).astype(int).tolist()
sample_output = pd.concat(
    [recommend_for_client(cid, top_n=3) for cid in sample_clients], ignore_index=True
)
sample_output


,ClientID,ProductID,Score,Type,Risk
0,3880,2,0.442067,0,0.30
1,3880,10,0.431809,0,0.13
2,3880,4,0.430778,0,0.44
3,742,6,0.955905,1,0.36
4,742,9,0.946882,1,0.27
5,742,5,0.946879,1,0.41
6,3152,2,0.386067,0,0.30
7,3152,10,0.378944,0,0.13
8,3152,3,0.377805,0,0.12
9,4160,6,0.958618,1,0.36


## Autoencoder

An **autoencoder-based recommender system** learns a non‑linear, low‑dimensional representation (embedding) of each user from their interaction vector with items. By compressing and then reconstructing the user–item interaction matrix, the model captures complex, non‑additive patterns that linear methods (e.g., SVD) may overlook. In this project, the autoencoder serves as an **alternative latent factor model** for product recommendation.

Let $R \in \mathbb{R}^{m \times n}$ be the user–item interaction matrix, where $m$ is the number of clients and $n$ the number of products. For a given client $u$, the input is the row vector $\mathbf{r}_u \in \mathbb{R}^n$.

The autoencoder consists of two functions:

- **Encoder** $f_\theta : \mathbb{R}^n \to \mathbb{R}^k$ with $k \ll n$ (bottleneck dimension)
- **Decoder** $g_\phi : \mathbb{R}^k \to \mathbb{R}^n$

The forward pass for user $u$ is:

$$
\mathbf{z}_u = f_\theta(\mathbf{r}_u) = \sigma(W_e \mathbf{r}_u + \mathbf{b}_e)
$$

$$
\hat{\mathbf{r}}_u = g_\phi(\mathbf{z}_u) = \sigma'(W_d \mathbf{z}_u + \mathbf{b}_d)
$$

The model is trained by minimising the **reconstruction loss** over all users:

$$
\mathcal{L} = \frac{1}{m} \sum_{u=1}^m \| \mathbf{r}_u - \hat{\mathbf{r}}_u \|_2^2 + \lambda \|\theta, \phi\|_2^2
$$

Once trained, the reconstructed vector $\hat{\mathbf{r}}_u$ contains **predicted scores for all products**, including those the client has not previously interacted with. The top‑$K$ products with the highest predicted scores are selected as recommendations.


### Autoencoder Architecture


In [ ]:
class RecommenderAutoencoder(nn.Module):
    def __init__(self, n_products, encoding_dim=10, dropout_rate=0.2):
        super().__init__()
        # Encoder: compress input vector to low-dimensional latent space
        self.encoder = nn.Sequential(
            nn.Linear(n_products, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Linear(32, encoding_dim),
        )
        # Decoder: reconstruct original vector from latent representation
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Dropout(dropout_rate),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Linear(64, n_products),
            nn.Sigmoid(),  # Ensure output scores are in [0, 1]
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

### Training loop


In [ ]:

# Convert interaction matrix to PyTorch tensor
X = torch.tensor(interaction_matrix.values, dtype=torch.float32)

model = RecommenderAutoencoder(n_products=X.shape[1], encoding_dim=8)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
criterion = nn.MSELoss()

epochs = 100
batch_size = 64
dataset = torch.utils.data.TensorDataset(X)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

for epoch in tqdm.tqdm(range(epochs), desc="Training Autoencoder"):
    epoch_loss = 0.0
    for batch in dataloader:
        inputs = batch[0]
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, inputs)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * inputs.size(0)
    avg_loss = epoch_loss / len(dataset)
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.6f}")

Training Autoencoder:  20%|██        | 20/100 [00:06<00:32,  2.50it/s]

Epoch 20/100, Loss: 0.002066


Training Autoencoder:  40%|████      | 40/100 [00:12<00:17,  3.53it/s]

Epoch 40/100, Loss: 0.001678


Training Autoencoder:  60%|██████    | 60/100 [00:18<00:16,  2.46it/s]

Epoch 60/100, Loss: 0.001455


Training Autoencoder:  80%|████████  | 80/100 [00:25<00:05,  3.77it/s]

Epoch 80/100, Loss: 0.001452


Training Autoencoder: 100%|██████████| 100/100 [00:31<00:00,  3.18it/s]

Epoch 100/100, Loss: 0.001167


### Generating Recommendations


In [ ]:
model.eval()
with torch.no_grad():
    reconstructed = model(X).numpy()

# For each client, retrieve top‑K product indices
k = 3
recommendations = []
for client_idx, row in enumerate(reconstructed):
    top_k_indices = row.argsort()[-k:][::-1]  # descending order
    for prod_idx in top_k_indices:
        recommendations.append(
            {
                "ClientID": interaction_matrix.index[client_idx],
                "ProductID": interaction_matrix.columns[prod_idx],
                "Score": row[prod_idx],
            }
        )
recommendations_df = pd.DataFrame(recommendations)

In [ ]:
recommendations_df.head()

,ClientID,ProductID,Score
0,0,9,0.927290
1,0,6,0.906924
2,0,5,0.888435
3,1,2,0.457692
4,1,10,0.447855
